# 🎙️ Khaledoon — Karma Listen Test (شرح كامل)
**الهدف:** تسمع صوت كارما مولَّدًا من النموذج المصري، وتفهم كل خطوة بتحصل.
**مفيش** رفع لـ Supabase، ولا لمس للطابور، ولا نشر. بس توليد ملف صوت على Colab تسمعه.

### الخطوات
1. من Colab: Runtime ثم Change runtime type ثم **GPU**.
2. املأ الخلية 🟢 (فيها 4 خانات، وجنب كل خانة «اعمل إيه»).
3. شغّل الخلايا اللي بعدها بالترتيب من فوق لتحت.
4. اسمع المشغّل اللي في آخر خلية، وقولّي سمعت إيه.

> الدفتر ده مبني على `Untitled8.ipynb`: خلايا التثبيت وتحميل النموذج الأصلية موجودة زي ما هي من غير حذف. مفيهوش أي جزء خاص بالراوي، لأن الدفتر الأصلي ماكانش فيه.

In [ ]:
#@title 🟢 الإعدادات — املأ هنا، ثم شغّل الخلايا اللي تحتها بالترتيب
import hashlib
import ipywidgets as widgets
from IPython.display import display, HTML
try:
    from google.colab import userdata
except Exception:
    userdata = None

# ثوابت مقفولة: نص مرجع كارما المكتوب هنا هو النص الحرفي للملف الصوتي المرجعي.
KARMA_REF_TEXT_DEFAULT = 'أنا كارما، ولما سمعت حكاية الدكتور مجدي يعقوب، فكرت في حاجة مهمة: إزاي طفل بيحب يساعد الناس ممكن يكبر ويبقى طبيبًا ينقذ حياة أطفال كتير؟'
KARMA_REF_TEXT_SHA256 = '50cce1612e8f55127d729c355b964e9d5335b21382572da3d96245ca37013346'

def _note(text):
    return widgets.HTML('<div dir="rtl" style="color:#555;font-size:13px;line-height:1.7;max-width:430px">' + text + '</div>')
def _title(text):
    return widgets.HTML('<div dir="rtl" style="font-weight:700;margin-top:10px">' + text + '</div>')
def _field(title, field, todo):
    return widgets.VBox([_title(title), widgets.HBox([field, _note('<b>اعمل إيه؟</b> ' + todo)], layout=widgets.Layout(flex_flow='row wrap', align_items='center', gap='12px'))])

w_ref_path = widgets.Text(value='/content/drive/MyDrive/Karma_Voice/assets_audio_science_magdi-yacoub_ar_000.mp3', layout=widgets.Layout(width='520px'))
w_ref_text = widgets.Textarea(value=KARMA_REF_TEXT_DEFAULT, rows=4, layout=widgets.Layout(width='520px'))
w_speak = widgets.Textarea(value='كل ما أسمع عن عالم قديم، أستغرب إزاي أفكاره ممكن تفضل عايشة بعده بمئات السنين. وابن سينا واحد من العلماء اللي حكايتهم تستحق نعرفها.', rows=4, layout=widgets.Layout(width='520px'))

try:
    _has_token = bool(userdata and userdata.get('KHALEDOON_WORKER_TOKEN'))
except Exception:
    _has_token = False
w_token = widgets.HTML('<div dir="rtl">' + ('✅ الرقم السري محفوظ في Colab' if _has_token else '⚠️ الرقم السري مش محفوظ في Colab') + '</div>')

display(HTML('<div dir="rtl" style="font-size:18px;font-weight:800">🟢 الإعدادات (4 خانات)</div>'))
display(_field('① مسار ملف مرجع كارما (الصوت الأصلي على Drive)', w_ref_path, 'سيبه زي ما هو. لو غيّرت مكان الملف في Drive، ضع المسار الجديد.'))
display(_field('② النص الحرفي لمرجع كارما (جاهز)', w_ref_text, 'متغيّرهوش. الدفتر هيرفض يكمّل لو اتغيّر حرف واحد، لأن غلطة هنا بتبوّظ التقليد.'))
display(_field('③ الجملة اللي عايز كارما تقولها', w_speak, 'اكتب الجملة اللي عايز تسمعها بصوت كارما، أو سيبها زي ما هي للتجربة.'))
display(_field('④ الرقم السري للعامل (Colab Secrets)', w_token, 'متعملش حاجة. ده بيتقرأ من الأسرار المحفوظة ومش بيتطبع ولا بيتبعت، والاختبار ده أصلًا مش بيستخدمه.'))
print('الإعدادات جاهزة. كمّل بتشغيل الخلايا اللي تحتها بالترتيب.')

### الخطوة 1 — تثبيت المكتبة (الخلية الأصلية)
بتثبّت مكتبة **OmniVoice** (محرك الصوت). بتاخد دقيقة أو اتنين. الخليتين دول من دفترك الأصلي زي ما هما (التانية تكرار بيعمل نفس الشيء).

In [ ]:
!pip install -q git+https://github.com/k2-fsa/OmniVoice.git
!pip install -q huggingface_hub soundfile

In [ ]:
!pip install -q git+https://github.com/k2-fsa/OmniVoice.git
!pip install -q huggingface_hub soundfile

### الخطوة 2 — تحميل النموذج (الخلية الأصلية)
بتنزّل أصوات مرجع جاهزة من مستودع `mohammedaly22/VoiceTut-TTS`، وتحمّل النموذج المصري `masterofaudio2077/voicetut-egyptian-tashkeel`. لو ظهرت `No module named 'omnivoice'` اعمل Runtime ثم Restart session وشغّل من الخطوة 1.

In [ ]:
import json
import torch
import soundfile as sf

from huggingface_hub import snapshot_download
from omnivoice import OmniVoice

refs = snapshot_download(
    "mohammedaly22/VoiceTut-TTS",
    allow_patterns=["reference_speakers/*"]
)

voice = json.load(
    open(f"{refs}/reference_speakers/references.json", encoding="utf-8")
)[0]

model = OmniVoice.from_pretrained(
    "masterofaudio2077/voicetut-egyptian-tashkeel",
    device_map="cuda:0" if torch.cuda.is_available() else "cpu",
    dtype=torch.float16 if torch.cuda.is_available() else torch.float32
)

print("VoiceTut Egyptian Tashkeel loaded successfully")

In [ ]:
# خلية: بيان الجهاز (بس للشرح، مفيهاش تعديل)
import torch
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('📌 الجهاز اللي شغال عليه الدفتر:', device)
print('   لو ظهر cpu، يبقى لازم تختار GPU من Runtime ثم Change runtime type.')
print('   صوت المرجع اللي اتحمّل فوق من HuggingFace (المتغير voice) هو صوت جاهز من مستودع VoiceTut ومش هنستخدمه هنا؛ هنستخدم صوت كارما الأصلي من Drive.')

### الخطوة 3 — تجهيز الصوت المرجعي والتأكد من الإعدادات
بتوصّل Drive (هيطلب إذنك)، وتتأكد إن ملف مرجع كارما موجود وإن نصه ما اتغيّرش.

In [ ]:
# خلية: تجهيز الصوت المرجعي والتأكد من الإعدادات
# حارس: لو الجلسة اتعمل لها Restart، المتغيرات بتتمسح. هنا بنتأكد إن الخطوات اللي قبل كده اتشغّلت.
_missing = [n for n in ('w_ref_path', 'w_ref_text', 'w_speak', 'KARMA_REF_TEXT_SHA256') if n not in globals()]
if _missing:
    raise RuntimeError('⛔ الخلية 🟢 (الإعدادات) لسه ماتشغّلتش في الجلسة دي، أو اتمسحت بعد Restart. شغّلها الأول (من فوق)، وبعدين رجّع شغّل الخلية دي.')
import os
from google.colab import drive
print('📌 اللي بيحصل: بنوصّل Google Drive عشان نقرا ملف مرجع كارما، وبعدين نتأكد إن الإعدادات سليمة.')
drive.mount('/content/drive')

ref_path = w_ref_path.value.strip()
ref_text = w_ref_text.value.strip()
speak_text = w_speak.value.strip()

if hashlib.sha256(ref_text.encode('utf-8')).hexdigest() != KARMA_REF_TEXT_SHA256:
    raise ValueError('⛔ نص مرجع كارما (الخانة ②) اتغيّر عن النص الأصلي. رجّع النص زي ما كان وشغّل الخلية 🟢 تاني.')
if not os.path.isfile(ref_path):
    raise FileNotFoundError('⛔ ملف مرجع كارما مش موجود على المسار ده: ' + ref_path + '  —  راجع الخانة ① وتأكد إن الملف فعلاً في Drive.')
if not speak_text:
    raise ValueError('⛔ الخانة ③ فاضية. اكتب الجملة اللي عايز كارما تقولها.')
print('✅ نص المرجع سليم، وملف المرجع موجود:', ref_path)
print('✅ الجملة اللي هتتقال:', speak_text)

### الخطوة 4 — توليد صوت كارما وتسمعه
بتحوّل المرجع، وتولّد جملتك بصوت كارما، وتعرض لك المرجع الأصلي والجديد جنب بعض عشان تقارن.

In [ ]:
# خلية: توليد صوت كارما وتشغيله
# حارس: لازم الخطوات اللي قبل كده تكون اتشغّلت في نفس الجلسة.
_missing = [n for n in ('model', 'device', 'ref_path', 'ref_text', 'speak_text') if n not in globals()]
if _missing:
    raise RuntimeError('⛔ في خطوات سابقة ماتشغّلتش في الجلسة دي (' + ', '.join(_missing) + '). شغّل الدفتر من فوق بالترتيب: الخلية 🟢 ثم التثبيت ثم تحميل النموذج ثم الخطوة 3، وبعدين الخلية دي.')
import subprocess, shutil
import numpy as np
import soundfile as sf
from IPython.display import Audio, display

print('📌 اللي بيحصل: (1) نحوّل ملف المرجع لصيغة WAV بسرعة 24 كيلوهرتز أحادي، (2) النموذج يقرا نص المرجع ويسمع الصوت المرجعي، (3) يولّد جملتك بنفس الصوت، (4) نعرضها لك تسمعها.')
if device == 'cpu':
    print('⚠️ الجهاز CPU مش GPU. هيشتغل بس هيبقى بطيء جدًا. الأفضل Runtime ثم Change runtime type ثم GPU.')
if not shutil.which('ffmpeg'):
    print('ffmpeg مش موجود، بنثبّته...')
    subprocess.run(['apt-get', '-qq', 'install', '-y', 'ffmpeg'], check=True)

ref_wav = '/content/karma_ref_24k.wav'
subprocess.run(['ffmpeg', '-y', '-i', ref_path, '-ac', '1', '-ar', '24000', '-c:a', 'pcm_s16le', ref_wav], check=True, capture_output=True)
print('🎧 ده صوت المرجع الأصلي (قارن بيه):')
display(Audio(ref_wav))

print('⏳ جاري التوليد... ممكن ياخد من نص دقيقة لكام دقيقة.')
generated = model.generate(
    text=speak_text,
    language='arz',
    ref_audio=ref_wav,
    ref_text=ref_text,
    num_step=36,
    guidance_scale=3.0,
    speed=0.97,
)[0]

audio = generated
if isinstance(generated, dict):
    audio = generated.get('audio', generated.get('waveform', generated.get('samples')))
if hasattr(audio, 'detach'):
    audio = audio.detach().cpu().numpy()
audio = np.asarray(audio, dtype=np.float32).squeeze()
if audio.size == 0 or not np.isfinite(audio).all():
    raise ValueError('⛔ الصوت اللي اتولّد فاضي أو فيه قيم غير صالحة.')

out_wav = '/content/karma_listen_test.wav'
sf.write(out_wav, audio, 24000)
print('✅ اتولّد الصوت. المدة تقريبًا:', round(audio.size / 24000, 2), 'ثانية. اتحفظ في:', out_wav)
print('🎧 اسمع كارما الجديدة:')
display(Audio(out_wav))
print('بعد ما تسمع، قولّي سمعت إيه بالظبط: مفهوم؟ شبه المرجع؟ في كلمات غلط؟ أو مفيش صوت؟')

### لو حصلت مشكلة
- `الخلية 🟢 لسه ماتشغّلتش` أو `name 'w_ref_path' is not defined` ← الجلسة اتعمل لها Restart أو الخلية 🟢 ماتشغّلتش. شغّل الدفتر من فوق بالترتيب.
- `No module named 'omnivoice'` ← Restart session وشغّل الخطوة 1 تاني.
- `الخانة ② اتغيّر` ← رجّع نص المرجع وشغّل الخلية 🟢 تاني.
- `ملف المرجع مش موجود` ← راجع الخانة ① ومكان الملف في Drive.
- الجهاز `cpu` ← اختار GPU من Runtime.
- الصوت غريب أو فيه كلمات غلط ← ماتغيّرش حاجة، وابعتلي بالظبط سمعت إيه.